# ML Hierarchical Clustering and PCA

This notebook documents the machine learning work for the hierarchical clustering part of Lab 7. The implementation and exploratory analysis were carried out in `ml/models/research/A7/clustering_h.ipynb`. The goal of the task was to:

- perform a clustering of the AIMO observations while excluding the score columns
- interpret the clusters
- perform a PCA of the AIMO observations
- draw the first two principal component scores in a colored 2D scatterplot
- interpret the PCA plot

## Dataset and preprocessing

The analysis used the processed AIMO dataset stored in `ml/models/data/processed/20190108_scores_and_weak_links.csv`.

Before clustering, the following preprocessing steps were applied:

1. The dataset was loaded from the processed CSV file.
2. Identifier and metadata columns (`ID` and `Date`) were removed.
3. All score-related columns were excluded. In this dataset this means that `SCORE` was removed.
4. The remaining movement-quality variables were used as the feature matrix for clustering and PCA.
5. The features were standardized with `StandardScaler` so that variables with larger numeric ranges would not dominate the distance calculations.

After preprocessing, the analysis was based on **2096 observations** and **14 movement variables**:

- `ForwardHead`
- `LeftArmFallForward`
- `RightArmFallForward`
- `LeftShoulderElevation`
- `RightShoulderElevation`
- `ExcessiveForwardLean`
- `LeftAsymmetricalWeightShift`
- `RightAsymmetricalWeightShift`
- `LeftKneeMovesInward`
- `RightKneeMovesInward`
- `LeftKneeMovesOutward`
- `RightKneeMovesOutward`
- `LeftHeelRises`
- `RightHeelRises`


## Hierarchical clustering method

The clustering was performed with **agglomerative hierarchical clustering** using **Ward linkage**. Ward linkage was chosen because it tends to merge observations in a way that keeps the within-cluster variance small, which usually gives compact and interpretable clusters in standardized numeric data.

The main steps were:

1. Standardize the feature matrix.
2. Compute the hierarchical linkage matrix with Ward linkage.
3. Plot the dendrogram to inspect the hierarchical structure.
4. Test several possible values of `k` by cutting the hierarchy into 2 to 7 clusters.
5. Compare the resulting clusterings using silhouette score.
6. Fit the final agglomerative clustering model using the selected number of clusters.
7. Summarize the resulting clusters using cluster sizes, mean feature values, and standardized deviations from the dataset average.

## Choosing the number of clusters

Several cluster cuts were evaluated. The silhouette score was used as a simple quantitative measure of cluster separation and compactness.

| k | Silhouette score | Smallest cluster | Largest cluster |
| - | ---------------- | ---------------- | --------------- |
| 2 | 0.2634 | 322 | 1774 |
| 7 | 0.2385 | 77 | 820 |
| 6 | 0.2228 | 86 | 820 |
| 5 | 0.1932 | 86 | 820 |
| 4 | 0.1726 | 322 | 820 |
| 3 | 0.1629 | 322 | 954 |

The best silhouette score was obtained for **k = 2**, so the final hierarchical clustering was reported using **two clusters**. The silhouette score is moderate rather than high, which means the clusters are not perfectly separated, but the split is still meaningful enough to identify one distinct subgroup in the data.

## Cluster results

The final cluster sizes were:

| Cluster | Number of observations |
| ------- | ---------------------- |
| 0 | 1774 |
| 1 | 322 |

A subset of the cluster mean values is shown below, focusing on the variables that contributed most to the interpretation:

| Cluster | ForwardHead | LeftArmFallForward | RightArmFallForward | RightKneeMovesOutward | LeftHeelRises | RightHeelRises |
| ------- | ----------- | ------------------ | ------------------- | --------------------- | ------------- | -------------- |
| 0 | 0.350 | 0.984 | 0.983 | 0.033 | 0.012 | 0.008 |
| 1 | 0.034 | 1.067 | 0.318 | 1.371 | 0.317 | 0.625 |

The strongest standardized deviations from the overall dataset mean were:

- **Cluster 0:** lower `RightKneeMovesOutward`, lower `RightHeelRises`, lower `LeftHeelRises`, and slightly higher `RightArmFallForward`
- **Cluster 1:** much higher `RightKneeMovesOutward`, higher `RightHeelRises`, higher `LeftHeelRises`, and lower `RightArmFallForward`

This shows that the second cluster is mainly separated by a lower-body movement pattern involving heel rise and outward knee movement, especially on the right side.

## PCA method

After clustering, PCA was applied to the **same standardized feature matrix**. This makes the PCA plot directly comparable to the clustering result.

The PCA procedure was:

1. Use the scaled AIMO observation matrix as input.
2. Compute the first two principal components.
3. Project each observation into the two-dimensional PCA space.
4. Plot the observations in a 2D scatterplot and color them by hierarchical cluster label.
5. Inspect the component loadings to understand which movement variables drive each axis.

## PCA results

The first two principal components explained the following amount of variance:

| Component | Explained variance ratio |
| --------- | ------------------------ |
| PC1 | 21.68% |
| PC2 | 14.77% |

Together, the first two components explained about **36.45%** of the total variance.

The main loadings for each component were:

| Component | Strongest loadings |
| --------- | ------------------ |
| PC1 | `RightHeelRises` (+), `RightKneeMovesOutward` (+), `LeftHeelRises` (+), `RightArmFallForward` (-), `ForwardHead` (-) |
| PC2 | `LeftArmFallForward` (+), `RightArmFallForward` (+), `RightShoulderElevation` (-), `LeftShoulderElevation` (-) |

This means that:

- moving to the right on **PC1** corresponds mainly to more heel rise and more outward knee movement
- moving to the left on **PC1** corresponds more to forward head and right arm fall-forward
- moving upward on **PC2** corresponds more to arm-fall-forward variables
- moving downward on **PC2** corresponds more to shoulder-elevation variables

## Interpretation

The hierarchical clustering divided the AIMO observations into two uneven groups: one large general cluster and one smaller, more distinct subgroup. The large cluster (**Cluster 0**) contains most of the observations and has relatively low values for `RightKneeMovesOutward`, `RightHeelRises`, and `LeftHeelRises`. This suggests that it represents the more common movement pattern in the dataset.

The smaller cluster (**Cluster 1**) is much more distinct. It has clearly higher values for outward knee movement and heel rise, especially on the right side. In particular, `RightKneeMovesOutward` was the strongest distinguishing feature for this group. Compared with the large cluster, this subgroup appears to represent observations with a more specific lower-body compensation pattern.

The PCA plot supports the clustering result. The two cluster centroids differ much more on **PC1** than on **PC2**, which means that the main separation between the clusters is horizontal in the score plot. Since PC1 is mostly driven by `RightHeelRises`, `RightKneeMovesOutward`, and `LeftHeelRises`, this confirms that the most important difference between the clusters is related to heel-rise and outward-knee behavior rather than shoulder variation.

At the same time, the first two principal components explain only about 36% of the total variance, so the PCA plot should be interpreted as a simplified 2D view of a higher-dimensional structure. The separation is therefore meaningful, but not complete. This matches the moderate silhouette score: the hierarchical clustering captures a real subgroup, but the overall dataset still contains overlap and variation inside the broader main cluster.

## Conclusion

The hierarchical clustering analysis found that a **2-cluster solution** was the most reasonable among the tested cuts of the dendrogram. The main result is that the AIMO observations contain:

- one large cluster with more typical movement values and lower heel-rise and outward-knee activity
- one smaller cluster with a clear lower-body movement pattern characterized by stronger heel rise and outward knee movement, especially on the right side

The PCA projection supports this interpretation by showing that the cluster separation is mainly along the first principal component, which is strongly associated with heel-rise and outward-knee variables. Overall, the hierarchical approach provides an interpretable grouping of the AIMO observations and highlights a meaningful subgroup that differs mainly in lower-body movement behavior.

## Reference to implementation notebook

The full implementation, including preprocessing, dendrogram plotting, silhouette-based cluster selection, cluster summary tables, PCA computation, and the colored PCA score plot, is available in:

- `ml/models/research/A7/clustering_h.ipynb`